# 証券営業インテリジェンス ハンズオン
## Part 1: Cortex AI セキュリティ・アクセス制御

このパートでは、金融機関において特に重要な **Snowflake Cortex AI のセキュリティ設定**を実践します。

### ハンズオン全体構成

| # | ノートブック | 内容 | 目安時間 |
|---|---|---|---|
| 事前準備 | `setup.sql` | DB・スキーマ・テーブル・データ生成・ウェアハウス作成 | 10分 |
| Part 1（本書） | `part1_security.ipynb` | Cortex AI セキュリティ設定・アクセス制御 | 30分 |
| Part 2 | `part2_ai_functions.ipynb` | AI Functions（要約・感情分析・抽出・分類・類似度） | 55分 |
| Part 3 | `part3_cortex_analyst.ipynb` | Semantic View 作成と Cortex Analyst（自然言語 to SQL） | 45分 |
| Part 4 | `part4_cortex_search.ipynb` | PDF 解析・Cortex Search Service 作成・セマンティック検索 | 60分 |
| Part 5 | `part5_cortex_agent.ipynb` | Cortex Agent 作成と Snowflake Intelligence デモ | 40分 |

### 前提条件
- `setup.sql` 実行済み（環境セットアップ・データ投入完了）

### 3層のアクセス制御

```
[Layer 1] CORTEX_ENABLED_CROSS_REGION  ─── データを処理するリージョンを制限
[Layer 2] CORTEX_MODELS_ALLOWLIST      ─── 使用可能なモデルをホワイトリストで管理
[Layer 3] Model RBAC (ロール別追加付与)  ─── Allowlist 外モデルを特定ロールに追加付与
```

### 🏦 金融機関（AWS利用）における推奨構成

> - AWSを利用している場合、Claude（Anthropic）モデルは **AWS US リージョン** で稼働
> - 日本リージョンのアカウントから Claude を使うには `CORTEX_ENABLED_CROSS_REGION = 'AWS_US'` が必要
> - 未承認モデルの使用を防ぐために `CORTEX_MODELS_ALLOWLIST` でホワイトリストを設定

> ⚠️ このノートブックの大部分は `ACCOUNTADMIN` 権限が必要です。

> ⏱️ **このパートの目安時間: 30分**

In [ ]:
%%sql -r result_env
USE ROLE ACCOUNTADMIN;
USE DATABASE SNOWFINANCE_DB;
USE SCHEMA DEMO_SCHEMA;
USE WAREHOUSE DEMO_WH;
SELECT CURRENT_ROLE() AS "ロール", CURRENT_REGION() AS "リージョン",
       CURRENT_DATABASE() AS DB, CURRENT_WAREHOUSE() AS WH;

## Layer 1: クロスリージョン推論の設定

### CORTEX_ENABLED_CROSS_REGION パラメータ

| 設定値 | 意味 | Claudeが使えるか？ |
|---|---|---|
| `DISABLED` | **デフォルト**。同一リージョンのモデルのみ | ❌（通常、日本リージョンにはない） |
| `ANY_REGION` | 全リージョンを許可 | ✅ |
| `AWS_US` | AWS 米国リージョンを追加許可 | ✅（Claudeは AWS Bedrock経由でUS稼働） |
| `AWS_EU` | AWS EUリージョンを追加許可 | 一部のみ |
| `AWS_US,AWS_EU` | 複数リージョンをカンマ区切りで指定 | ✅ |

> 💡 **AWS利用金融機関向けの推奨設定**: `CORTEX_ENABLED_CROSS_REGION = 'AWS_US'`
> データはAWS内ネットワーク（暗号化済み）のみを経由し、公衆インターネットを通りません。

> ⏱️ **目安: 4分**

In [ ]:
%%sql -r result_cross_region
-- 現在のクロスリージョン設定を確認
SHOW PARAMETERS LIKE 'CORTEX_ENABLED_CROSS_REGION' IN ACCOUNT;

In [ ]:
%%sql -r result_set_cross_region
-- ハンズオン用: ANY_REGION に設定（全モデルを使用可能にする）
-- 本番環境では 'AWS_US' または 'DISABLED' を推奨
ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'ANY_REGION';

-- 設定確認
SHOW PARAMETERS LIKE 'CORTEX_ENABLED_CROSS_REGION' IN ACCOUNT;

## Layer 2: モデルホワイトリスト (CORTEX_MODELS_ALLOWLIST)

アカウント全体で**使用可能なモデルをホワイトリストで制限**できます。
未承認モデルの使用をシステム的に防ぎ、コスト管理とガバナンスを強化します。

```sql
-- 全モデルを許可（デフォルト）
ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'All';

-- 特定モデルのみ許可（金融機関向け推奨）
ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'claude-sonnet-4-6,claude-opus-4-6';

-- 全モデルを禁止（Model RBAC のみで制御する場合）
ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'None';
```

> ⏱️ **目安: 4分**

In [ ]:
%%sql -r result_allowlist_check
-- 現在のモデルホワイトリスト設定を確認
SHOW PARAMETERS LIKE 'CORTEX_MODELS_ALLOWLIST' IN ACCOUNT;

In [ ]:
%%sql -r result_set_allowlist
-- ハンズオン用: Claudeモデルのみをホワイトリストに設定
ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'claude-sonnet-4-6,claude-opus-4-6';

-- 設定確認
SHOW PARAMETERS LIKE 'CORTEX_MODELS_ALLOWLIST' IN ACCOUNT;

In [ ]:
-- ホワイトリスト外モデルを試みる（エラーになることを確認）
-- ※ ACCOUNTADMIN はホワイトリスト制限をバイパスするため、制限付きロールで検証
-- ※ 初回実行時は Layer 3 でロールを作成してから戻ってください
USE ROLE DEMO_ANALYST_ROLE;
USE SECONDARY ROLES NONE;

SELECT AI_COMPLETE('openai-gpt-5', '日本経済について一言') AS "応答";
-- -> Error: Model does not exist or is not authorized.

In [ ]:
-- ホワイトリスト内モデルは正常動作
SELECT AI_COMPLETE('claude-sonnet-4-6', 'Snowflakeを50文字で表現して') AS "応答";

In [ ]:
-- ハンズオン用: ホワイトリストを元に戻す（全モデル許可）
USE ROLE ACCOUNTADMIN;
ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'All';
SHOW PARAMETERS LIKE 'CORTEX_MODELS_ALLOWLIST' IN ACCOUNT;

## Layer 3: Model RBAC（ロール別モデル追加付与）

### 設計思想: 最小権限の原則

```
【全社デフォルト: Allowlist で安全な基準線を設定】
CORTEX_MODELS_ALLOWLIST = 'claude-sonnet-4-6'
→ 全員が使えるのは claude-sonnet-4-6 のみ

【上位ロールへの追加付与: Model RBAC で例外を承認】
GRANT CORTEX-MODEL-ROLE-CLAUDE-OPUS-4-6 TO ROLE 上級アナリスト
→ 承認された役割だけが高性能モデル（Opus）も使える
```

### CORTEX_MODELS_ALLOWLIST との関係

| 制御方法 | 範囲 | 用途 |
|---|---|---|
| `CORTEX_MODELS_ALLOWLIST` | アカウント全体 | 会社標準モデルを定義・全社制限 |
| `CORTEX-MODEL-ROLE-*` | ロール単位 | Allowlist 外モデルを特定ロールに追加付与 |

> 💡 **金融機関向けベストプラクティス**:
> Allowlist で安全なデフォルトを設定し、Model RBAC で上位ロールにのみ高性能モデルを付与。
> 「デフォルト制限 → 承認ベースでエスカレーション」が最小権限の原則に沿った構成です。

> ⏱️ **目安: 7分**

> ✅ `CORTEX_BASE_MODELS_REFRESH()` は `setup.sql` で実行済みです。モデルオブジェクトはすでに `SNOWFLAKE.MODELS` スキーマに生成されています。

In [ ]:
%%sql -r result_show_models
-- Step 1: 利用可能なモデルオブジェクトを確認（setup.sql で REFRESH 済み）
SHOW MODELS IN SNOWFLAKE.MODELS;

In [ ]:
%%sql -r result_app_roles
-- Claudeモデルに対応したアプリケーションロールを確認
SHOW APPLICATION ROLES IN APPLICATION SNOWFLAKE;

In [ ]:
%%sql -r result_create_roles
-- Step 3: 2種類のデモロールを作成
USE ROLE ACCOUNTADMIN;

-- 1. 通常アナリストロール（Allowlist 範囲内のみ使用可能）
CREATE ROLE IF NOT EXISTS DEMO_ANALYST_ROLE;
GRANT USAGE ON DATABASE SNOWFINANCE_DB TO ROLE DEMO_ANALYST_ROLE;
GRANT USAGE ON SCHEMA SNOWFINANCE_DB.DEMO_SCHEMA TO ROLE DEMO_ANALYST_ROLE;
GRANT USAGE ON WAREHOUSE DEMO_WH TO ROLE DEMO_ANALYST_ROLE;
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE DEMO_ANALYST_ROLE;

-- 2. 上級アナリストロール（Allowlist 外の claude-opus-4-6 も Model RBAC で追加付与）
CREATE ROLE IF NOT EXISTS DEMO_SENIOR_ANALYST_ROLE;
GRANT USAGE ON DATABASE SNOWFINANCE_DB TO ROLE DEMO_SENIOR_ANALYST_ROLE;
GRANT USAGE ON SCHEMA SNOWFINANCE_DB.DEMO_SCHEMA TO ROLE DEMO_SENIOR_ANALYST_ROLE;
GRANT USAGE ON WAREHOUSE DEMO_WH TO ROLE DEMO_SENIOR_ANALYST_ROLE;
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE DEMO_SENIOR_ANALYST_ROLE;
GRANT APPLICATION ROLE SNOWFLAKE."CORTEX-MODEL-ROLE-CLAUDE-OPUS-4-6"
    TO ROLE DEMO_SENIOR_ANALYST_ROLE;

SELECT '2種類のデモロール作成完了' AS STATUS;

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

# Step 4a: Allowlist を会社標準（claude-sonnet-4-6 のみ）に設定
# ※ SQLで直接実行する場合（<ユーザー名> を自分のユーザー名に変更してください）:
#   USE ROLE ACCOUNTADMIN;
#   ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'claude-sonnet-4-6';
#   GRANT ROLE DEMO_ANALYST_ROLE TO USER "<ユーザー名>";
#   USE ROLE DEMO_ANALYST_ROLE;

session.sql("USE ROLE ACCOUNTADMIN").collect()
session.sql("ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'claude-sonnet-4-6'").collect()

current_user = session.sql("SELECT CURRENT_USER()").collect()[0][0]
session.sql(f'GRANT ROLE DEMO_ANALYST_ROLE TO USER "{current_user}"').collect()
print(f"✅ DEMO_ANALYST_ROLE を {current_user} に付与完了")

session.sql("USE ROLE DEMO_ANALYST_ROLE").collect()
session.sql("USE SECONDARY ROLES NONE").collect()

result = session.sql(
    "SELECT AI_COMPLETE('claude-sonnet-4-6', 'Snowflakeを一言で説明して') AS \"標準モデル_応答\""
).collect()
print(f"\n【標準モデル_応答】\n{result[0][0]}")


In [ ]:
-- Step 4b: 通常アナリストロールで Allowlist 外モデルを試みる
USE ROLE DEMO_ANALYST_ROLE;
USE SECONDARY ROLES NONE;

-- Allowlist 外モデル + Model RBAC 未付与 -> エラーになるはず
SELECT AI_COMPLETE(
    'claude-opus-4-6',
    '高度な分析をお願いします'
) AS "上位モデル_応答";
-- -> Error: Model does not exist or is not authorized.

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

# Step 4c: 上級アナリストロールに切り替え -> Opus も使えることを確認
# ※ SQLで直接実行する場合（<ユーザー名> を自分のユーザー名に変更してください）:
#   USE ROLE ACCOUNTADMIN;
#   GRANT ROLE DEMO_SENIOR_ANALYST_ROLE TO USER "<ユーザー名>";
#   USE ROLE DEMO_SENIOR_ANALYST_ROLE;

session.sql("USE ROLE ACCOUNTADMIN").collect()
current_user = session.sql("SELECT CURRENT_USER()").collect()[0][0]
session.sql(f'GRANT ROLE DEMO_SENIOR_ANALYST_ROLE TO USER "{current_user}"').collect()
print(f"✅ DEMO_SENIOR_ANALYST_ROLE を {current_user} に付与完了")

session.sql("USE ROLE DEMO_SENIOR_ANALYST_ROLE").collect()
session.sql("USE SECONDARY ROLES NONE").collect()

result = session.sql(
    "SELECT AI_COMPLETE('claude-opus-4-6', '高度な分析をお願いします') AS \"上位モデル_応答\""
).collect()
print(f"\n【上位モデル_応答】\n{result[0][0]}")


In [ ]:
-- Step 4d: ハンズオン用クリーンアップ -> Allowlist を元に戻す
USE ROLE ACCOUNTADMIN;
ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'All';

SHOW GRANTS TO ROLE DEMO_SENIOR_ANALYST_ROLE;

## まとめ

Part 1 完了！3層のセキュリティ設定を実践しました。

### セキュリティ設定サマリー

| Layer | パラメータ / 機能 | 今回の設定 | 本番推奨 |
|---|---|---|---|
| 1 | `CORTEX_ENABLED_CROSS_REGION` | `ANY_REGION`（ハンズオン用） | `AWS_US`（Claudeを使うなら必須） |
| 2 | `CORTEX_MODELS_ALLOWLIST` | `All`（元に戻した） | `claude-sonnet-4-6,claude-opus-4-6` など |
| 3 | Model RBAC | Allowlist + DEMO_SENIOR_ANALYST_ROLE に Opus 追加付与 | Allowlist で制限 + Model RBAC でエスカレーション |

### 設定コマンドリファレンス

```sql
-- Layer 1: クロスリージョン
ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'AWS_US';  -- AWSのみ利用金融機関向け

-- Layer 2: モデルホワイトリスト
ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'claude-sonnet-4-6,claude-opus-4-6';

-- Layer 3: Model RBAC
CALL SNOWFLAKE.MODELS.CORTEX_BASE_MODELS_REFRESH();
-- Allowlist で全社制限
ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'claude-sonnet-4-6';
-- 上位ロールに高性能モデルを追加付与
GRANT APPLICATION ROLE SNOWFLAKE."CORTEX-MODEL-ROLE-CLAUDE-OPUS-4-6" TO ROLE SENIOR_ROLE;

```

**次のステップ:** `part2_ai_functions.ipynb` で Cortex AI Functions の実践的な活用を体験してください。

In [ ]:
%%sql -r result_restore_role
-- テスト後は ACCOUNTADMIN に戻す
USE ROLE ACCOUNTADMIN;
SELECT 'ACCOUNTADMIN に戻しました' AS STATUS;